# QSSL Stage 2 — Colab GPU Runner

Runs the QSSL stage2_bench experiments (5 seeds × 50 epochs × 10 000 graphs) on Colab GPU.
Results are logged to W&B project `team-sanya/qssl-experiments`.

**Before running:**
1. Runtime → Change runtime type → **GPU** (T4 or A100)
2. Add your W&B API key as a Colab secret named `WANDB_API_KEY`
   (left sidebar → key icon → add secret)

**Stage 2 hard set** (promoted from stage1):
- `H8_qc1_barlow` — best AUC + lowest variance
- `H1_qc1_pairs` — primary quantum baseline
- `G0_classical_baseline` — classical reference
- `H11_qc1_noisyB` — NISQ noise robustness
- `G1_param_matched_mlp` — capacity falsification control

**GPU-only set** (too slow on CPU, run here):
- `H3_qc3_pairs` — QC3 amplitude embedding
- `H4_qc4_reuploading` — data re-uploading (72 params)


## 1. Check GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No GPU found — change runtime type to GPU')

## 2. Install dependencies

In [ ]:
import subprocess

# --- Step 1: pin numpy 1.26.4 FIRST ---
# PennyLane C extensions require numpy 1.x ABI.
# torch wheels can pull numpy 2.x → "dtype size changed" error.
!pip install -q 'numpy==1.26.4'

# --- Step 2: detect CUDA version ---
r = subprocess.run(['nvcc', '--version'], capture_output=True, text=True)
cuda_ver = 'cu121'  # Colab T4/A100 ship CUDA 12.x
if 'release 11' in r.stdout:
    cuda_ver = 'cu118'
print(f'CUDA variant: {cuda_ver}')

# --- Step 3: torch + torch_geometric ---
!pip install -q torch==2.2.0 torchvision --index-url https://download.pytorch.org/whl/{cuda_ver}
!pip install -q torch_geometric
!pip install -q pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv \
    -f https://data.pyg.org/whl/torch-2.2.0+{cuda_ver}.html

# --- Step 4: PennyLane 0.42 + pinned autoray ---
# - pennylane 0.38 imports jax.core.Primitive removed in JAX >= 0.4.25.
# - pennylane 0.42 fixes that, but autoray 0.7+ removed NumpyMimic which
#   pennylane/math/__init__.py still imports → AttributeError at import time.
# - Pin autoray==0.6.12 to stay on the 0.6.x API that pennylane 0.42 expects.
# - We use default.qubit (not Lightning), so pennylane-lightning is not needed.
!pip install -q 'pennylane==0.42.0' 'autoray==0.6.12'

# --- Step 5: W&B and misc ---
!pip install -q wandb wandb-workspaces scikit-learn matplotlib

# --- Step 6: re-enforce numpy pin (transitive deps may have re-pulled 2.x) ---
!pip install -q 'numpy==1.26.4'

print('\nAll dependencies installed.')
print('Restarting kernel so the pinned numpy is what gets imported...')
import IPython
IPython.Application.instance().kernel.do_shutdown(restart=True)

## 3. W&B login

In [ ]:
import wandb
from google.colab import userdata

WANDB_API_KEY = userdata.get('WANDB_API_KEY')
wandb.login(key=WANDB_API_KEY)
print('Logged in to W&B.')

## 4. Clone repo

In [ ]:
import os

REPO_URL  = 'https://github.com/SanyaNanda/ML4Sci_QuantumContrastiveLearning.git'
REPO_DIR  = '/content/qssl'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
## 4b. Apply patches (fixes not yet in GitHub main)
from pathlib import Path
repo = Path(REPO_DIR)

# --- Fix 1: BatchNorm import in qgnn.py ---
# GitHub has: from torch.nn import ..., BatchNorm   (wrong — torch.nn has no BatchNorm)
# Fix moves BatchNorm to torch_geometric.nn where it belongs.
qgnn_path = repo / 'qssl/models/qgnn.py'
text = qgnn_path.read_text()
old_import = 'from torch.nn import Linear, ReLU, Sigmoid, ModuleList, LeakyReLU, Linear, BatchNorm1d, BatchNorm'
new_import = 'from torch.nn import Linear, ReLU, Sigmoid, ModuleList, LeakyReLU, BatchNorm1d'
old_pyg    = 'from torch_geometric.nn import GATConv, global_mean_pool, global_max_pool'
new_pyg    = 'from torch_geometric.nn import GATConv, global_mean_pool, global_max_pool, BatchNorm'
if old_import in text:
    text = text.replace(old_import, new_import).replace(old_pyg, new_pyg)
    qgnn_path.write_text(text)
    print('✓ Fixed BatchNorm import in qgnn.py')
else:
    print('  qgnn.py already patched or differs — skipping')

# --- Fix 2: conditional TF import in losses.py ---
losses_path = repo / 'qssl/loss/losses.py'
text = losses_path.read_text()
if 'import tensorflow as tf\nfrom qssl.config import Config' in text:
    text = text.replace(
        'import tensorflow as tf\nfrom qssl.config import Config',
        'from qssl.config import Config\ntry:\n    import tensorflow as tf\n    from tensorflow import keras\n    _TF_AVAILABLE = True\nexcept ImportError:\n    _TF_AVAILABLE = False'
    )
    losses_path.write_text(text)
    print('✓ Fixed conditional TF import in losses.py')
else:
    print('  losses.py already patched or differs — skipping')

print('Patches applied.')


## 5. Download QG graph data from W&B artifact

In [ ]:
import os
from pathlib import Path

api = wandb.Api()
artifact = api.artifact('team-sanya/qssl-experiments/qg-graph-data:latest')
artifact_dir = Path(artifact.download())   # downloads to ./artifacts/qg-graph-data:v.../
print(f'Artifact downloaded to: {artifact_dir}')
print('Files:', list(artifact_dir.glob('**/*')))

# The data loader reads from REPO_ROOT/data/qg_graph/
# Symlink the artifact subfolder to that expected path.
data_dir = Path(REPO_DIR) / 'data' / 'qg_graph'
data_dir.parent.mkdir(parents=True, exist_ok=True)

src = artifact_dir / 'qg_graph'
if not data_dir.exists():
    os.symlink(src, data_dir)
    print(f'Symlinked {src} -> {data_dir}')
else:
    print(f'{data_dir} already exists, skipping symlink.')

# Verify
assert (data_dir / 'x10_sorted_12500.npy').exists(), 'x file missing!'
assert (data_dir / 'y10_sorted_12500.npy').exists(), 'y file missing!'
print('Data files verified.')

## 6. Verify torch + CUDA + PennyLane

In [ ]:
import torch
import pennylane as qml

print(f'torch:      {torch.__version__}')
print(f'CUDA:       {torch.cuda.is_available()} — {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"}')
print(f'pennylane:  {qml.__version__}')

# Quick PennyLane smoke test
dev = qml.device('default.qubit', wires=2)
@qml.qnode(dev)
def test_circuit(x):
    qml.RX(x, wires=0)
    qml.CNOT(wires=[0, 1])
    return qml.expval(qml.PauliZ(0))

import torch
x = torch.tensor(0.5, requires_grad=True)
out = test_circuit(x)
print(f'PennyLane test circuit output: {out:.4f}  (expected ~0.8776)')

## 7. Configure experiment run

Edit `EXPERIMENTS_TO_RUN` to select which experiments to run.
The default is the **hard stage2 set** (promoted from stage1).
Uncomment the GPU-only set to also run H3/H4.

In [ ]:
# --- Stage 2 configuration ---
STAGE           = 'stage2_bench'   # 10 000 graphs, 50 epochs
SEEDS           = [0, 1, 13, 42, 123]
EPOCHS          = 50
BATCH_SIZE      = 128              # GPU has enough VRAM for full batch
WANDB_PROJECT   = 'qssl-experiments'
WANDB_MODE      = 'online'

# Hard stage2 set (all CPU-feasible circuits, promoted from stage1)
EXPERIMENTS_TO_RUN = [
    'H8_qc1_barlow',          # Best AUC + lowest variance
    'H1_qc1_pairs',           # Primary quantum baseline
    'G0_classical_baseline',  # Classical reference
    'H11_qc1_noisyB',         # NISQ noise robustness
    'G1_param_matched_mlp',   # Capacity falsification control
]

# GPU-only set — uncomment to include
# EXPERIMENTS_TO_RUN += [
#     'H3_qc3_pairs',          # QC3 amplitude embedding (24 params, ~4 s/epoch on T4)
#     'H4_qc4_reuploading',    # Data re-uploading (72 params, ~15 s/epoch on T4)
#     'H5_qc6_HEA',            # HEA (48 params, ~8 s/epoch on T4)
# ]

print(f'Will run: {EXPERIMENTS_TO_RUN}')
print(f'Stage: {STAGE} | Seeds: {SEEDS} | Epochs: {EPOCHS} | Batch size: {BATCH_SIZE}')

## 8. Run stage 2 experiments

In [ ]:
import sys, json
from pathlib import Path

sys.path.insert(0, REPO_DIR)

# Import the registry and trainer from the repo
from scripts.run_experiments import (
    default_registry, StageConfig, train_one, ExperimentConfig
)
import dataclasses
import numpy as np

registry = {c.name: c for c in default_registry()}

stage2 = StageConfig(
    name=STAGE,
    n_train=10_000,
    n_val=1_000,
    n_test=1_500,
    epochs=EPOCHS,
    seeds=tuple(SEEDS),
    promotion_slack=0.01,
)

all_results = {}

for exp_name in EXPERIMENTS_TO_RUN:
    cfg = registry[exp_name]
    # Apply batch size override
    cfg = dataclasses.replace(cfg, batch_size=BATCH_SIZE)

    seed_metrics = []
    print(f'\n{"="*60}')
    print(f'Running {exp_name}')
    print(f'{"="*60}')

    for seed in SEEDS:
        print(f'  seed {seed} ...')
        try:
            m = train_one(cfg, stage2, seed, WANDB_PROJECT, WANDB_MODE)
            seed_metrics.append(m)
            print(f'    AUC={m["test_auc"]:.4f}  acc={m["test_accuracy"]:.4f}')
        except Exception as e:
            print(f'    FAILED: {e}')
            seed_metrics.append({'error': str(e), 'seed': seed})

    good = [m for m in seed_metrics if 'error' not in m]
    if good:
        aucs = np.array([m['test_auc'] for m in good])
        accs = np.array([m['test_accuracy'] for m in good])
        summary = dict(
            name=exp_name,
            n_seeds=len(good),
            mean_auc=float(aucs.mean()),
            std_auc=float(aucs.std(ddof=0) if len(aucs) > 1 else 0.0),
            mean_acc=float(accs.mean()),
            std_acc=float(accs.std(ddof=0) if len(accs) > 1 else 0.0),
        )
        all_results[exp_name] = summary

        # Save per-experiment summary
        out_dir = Path(REPO_DIR) / 'experiments' / exp_name
        out_dir.mkdir(parents=True, exist_ok=True)
        (out_dir / 'summary_stage2.json').write_text(json.dumps(summary, indent=2))

        print(f'\n  => {exp_name}: mean AUC {summary["mean_auc"]:.4f} +/- {summary["std_auc"]:.4f}')

print('\nAll experiments complete.')

## 9. Stage 2 leaderboard

In [ ]:
if all_results:
    print(f'{"Experiment":<30} {"Mean AUC":>10} {"Std AUC":>10} {"Mean Acc":>10}')
    print('-' * 65)
    for name, r in sorted(all_results.items(), key=lambda x: -x[1]['mean_auc']):
        print(f'{name:<30} {r["mean_auc"]:>10.4f} {r["std_auc"]:>10.4f} {r["mean_acc"]:>10.4f}')
else:
    print('No results yet.')

## 10. Save trained models as .pkl files

Per the experiment plan: all stage2 trained encoders are saved as `.pkl` files
alongside the `.pt` PyTorch checkpoints.

In [ ]:
import pickle, glob
from pathlib import Path

repo = Path(REPO_DIR)
pt_files = list(repo.glob('experiments/*/model_seed*.pt'))

if not pt_files:
    print('No .pt model files found — they are saved by train_one() inside the trainer.')
    print('Check experiments/<name>/model_seed<seed>.pt')
else:
    for pt_path in pt_files:
        import torch
        pkl_path = pt_path.with_suffix('.pkl')
        state_dict = torch.load(pt_path, map_location='cpu')
        with open(pkl_path, 'wb') as f:
            pickle.dump(state_dict, f)
        print(f'Saved {pkl_path.name}  ({pkl_path.stat().st_size/1e6:.2f} MB)')

print('Done.')

## 11. Upload stage 2 models as W&B artifact

In [ ]:
import wandb
from pathlib import Path

repo = Path(REPO_DIR)
run = wandb.init(
    project=WANDB_PROJECT,
    entity='team-sanya',
    job_type='model-upload',
    name='stage2-model-artifacts',
)

artifact = wandb.Artifact(
    name='qssl-stage2-models',
    type='model',
    description='Stage 2 trained encoder weights (.pt and .pkl) for all promoted experiments.',
)

added = 0
for exp_name in EXPERIMENTS_TO_RUN:
    exp_dir = repo / 'experiments' / exp_name
    for ext in ('*.pt', '*.pkl', 'summary_stage2.json'):
        for f in exp_dir.glob(ext):
            artifact.add_file(str(f), name=f'{exp_name}/{f.name}')
            added += 1

if added:
    run.log_artifact(artifact)
    print(f'Uploaded {added} files as artifact qssl-stage2-models.')
else:
    print('No model files found to upload.')

run.finish()

## 12. Research question answers (fill in after results are in)

| Research Question | Answer | Key Evidence |
|---|---|---|
| Circuit choice at fixed depth/qubits? | TBD | H1 vs H3 vs H4 stage2 AUC |
| Quantum capacity vs classical? | TBD | G1 vs H1 stage2 AUC |
| Loss geometry helping? | TBD | H10 vs H1F (needs torch impl) |
| Fidelity vs pairwise? | Pairs preferred (stage1) | H1F 0.5703 << H1 0.6777 |
| NISQ noise tolerance? | TBD | H11 vs H1 stage2 |
| IBM hardware gap? | TBD | Track C — requires IBMQ_TOKEN |

**Next step after stage2:** Run IBM Quantum hardware evaluation (Track C).
See `experiment_plan.md` section *Post-stage pipeline* for the full protocol.